In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Input,
    BatchNormalization
)
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

import os
import joblib

In [5]:
df = pd.read_csv("../data/processed_data/feature_engineered_dataset.csv")

print("Dataset Loaded Successfully")
print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])

Dataset Loaded Successfully
DATASET LOADED
Rows    : 68036
Columns : 15


In [6]:
X = df.drop("Heart_Disease", axis=1)
y = df["Heart_Disease"]

print("=" * 60)
print("FEATURES AND TARGET")
print("=" * 60)

print("Number of Features :", X.shape[1])
print("Target             : Heart_Disease")

print("\nTarget Distribution:")
print(y.value_counts())

FEATURES AND TARGET
Number of Features : 14
Target             : Heart_Disease

Target Distribution:
Heart_Disease
0    34491
1    33545
Name: count, dtype: int64


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("=" * 60)
print("TRAIN-TEST SPLIT")
print("=" * 60)

print("Training Samples :", X_train.shape[0])
print("Testing Samples  :", X_test.shape[0])

TRAIN-TEST SPLIT
Training Samples : 54428
Testing Samples  : 13608


In [8]:
# ---------------------------------------------
# Feature Scaling using RobustScaler
# ---------------------------------------------

scaler = RobustScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("=" * 60)
print("ROBUST SCALING COMPLETED")
print("=" * 60)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

ROBUST SCALING COMPLETED
Training Shape : (54428, 14)
Testing Shape  : (13608, 14)


In [9]:
# ==========================================================
# STEP 5 - HYPERPARAMETER TUNING
# Function to create ANN models
# ==========================================================

def create_ann(
    input_features,
    neurons=(128, 64, 32),
    dropout_rate=0.25,
    learning_rate=0.0005
):
    
    model = Sequential()

    # Input Layer
    model.add(Input(shape=(input_features,)))

    # Hidden Layer 1
    model.add(Dense(neurons[0], activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # Hidden Layer 2
    model.add(Dense(neurons[1], activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    # Hidden Layer 3
    model.add(Dense(neurons[2], activation="relu"))

    # Output Layer
    model.add(Dense(1, activation="sigmoid"))

    # Compile
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

print("ANN model creation function is ready.")

ANN model creation function is ready.


In [10]:
# ==========================================================
# Training Callbacks
# ==========================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

print("Training callbacks are ready.")

Training callbacks are ready.


In [11]:
# ==========================================================
# Hyperparameter Configurations
# ==========================================================

experiments = [

    {
        "name": "Experiment 1",
        "neurons": (128, 64, 32),
        "dropout": 0.20,
        "learning_rate": 0.0005,
        "batch_size": 32
    },

    {
        "name": "Experiment 2",
        "neurons": (128, 64, 32),
        "dropout": 0.25,
        "learning_rate": 0.0005,
        "batch_size": 32
    },

    {
        "name": "Experiment 3",
        "neurons": (128, 64, 32),
        "dropout": 0.30,
        "learning_rate": 0.0005,
        "batch_size": 32
    },

    {
        "name": "Experiment 4",
        "neurons": (128, 64, 32),
        "dropout": 0.25,
        "learning_rate": 0.001,
        "batch_size": 32
    },

    {
        "name": "Experiment 5",
        "neurons": (128, 64, 32),
        "dropout": 0.25,
        "learning_rate": 0.00025,
        "batch_size": 32
    },

    {
        "name": "Experiment 6",
        "neurons": (256, 128, 64),
        "dropout": 0.25,
        "learning_rate": 0.0005,
        "batch_size": 32
    },

    {
        "name": "Experiment 7",
        "neurons": (128, 64, 32),
        "dropout": 0.25,
        "learning_rate": 0.0005,
        "batch_size": 64
    }
]

print("Number of experiments:", len(experiments))

Number of experiments: 7


In [12]:
# ==========================================================
# RUN HYPERPARAMETER EXPERIMENTS
# ==========================================================

results = []

best_accuracy = 0
best_model = None
best_experiment = None

for experiment in experiments:

    print("\n" + "=" * 70)
    print(experiment["name"])
    print("=" * 70)

    # Create model
    model = create_ann(
        input_features=X_train.shape[1],
        neurons=experiment["neurons"],
        dropout_rate=experiment["dropout"],
        learning_rate=experiment["learning_rate"]
    )

    # Fresh callbacks
    early_stopping_exp = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    reduce_lr_exp = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=0
    )

    # Train
    history = model.fit(
        X_train,
        y_train,
        validation_split=0.20,
        epochs=100,
        batch_size=experiment["batch_size"],
        callbacks=[
            early_stopping_exp,
            reduce_lr_exp
        ],
        verbose=0
    )

    # Predict
    y_prob = model.predict(
        X_test,
        verbose=0
    ).ravel()

    y_pred = (y_prob >= 0.5).astype(int)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store results
    results.append({
        "Experiment": experiment["name"],
        "Neurons": str(experiment["neurons"]),
        "Dropout": experiment["dropout"],
        "Learning Rate": experiment["learning_rate"],
        "Batch Size": experiment["batch_size"],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc
    })

    # Keep actual best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = model
        best_experiment = experiment.copy()

    print(f"Accuracy  : {accuracy * 100:.2f}%")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"ROC AUC   : {roc_auc:.4f}")


print("\n" + "=" * 70)
print("BEST MODEL STORED")
print("=" * 70)

print("Best Experiment :", best_experiment["name"])
print("Best Accuracy   :", f"{best_accuracy * 100:.2f}%")


Experiment 1
Accuracy  : 73.41%
Precision : 0.7508
Recall    : 0.6894
F1 Score  : 0.7188
ROC AUC   : 0.8019

Experiment 2
Accuracy  : 73.43%
Precision : 0.7495
Recall    : 0.6925
F1 Score  : 0.7199
ROC AUC   : 0.8018

Experiment 3
Accuracy  : 73.28%
Precision : 0.7504
Recall    : 0.6864
F1 Score  : 0.7170
ROC AUC   : 0.8009

Experiment 4
Accuracy  : 73.30%
Precision : 0.7552
Recall    : 0.6783
F1 Score  : 0.7147
ROC AUC   : 0.8016

Experiment 5
Accuracy  : 73.55%
Precision : 0.7516
Recall    : 0.6924
F1 Score  : 0.7208
ROC AUC   : 0.8014

Experiment 6
Accuracy  : 73.46%
Precision : 0.7475
Recall    : 0.6973
F1 Score  : 0.7215
ROC AUC   : 0.8012

Experiment 7
Accuracy  : 73.52%
Precision : 0.7538
Recall    : 0.6876
F1 Score  : 0.7192
ROC AUC   : 0.8021

BEST MODEL STORED
Best Experiment : Experiment 5
Best Accuracy   : 73.55%


In [13]:
# ==========================================================
# COMPARE ALL EXPERIMENTS
# ==========================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

results_df.reset_index(drop=True, inplace=True)

print("=" * 60)
print("HYPERPARAMETER EXPERIMENT COMPARISON")
print("=" * 60)

results_df

HYPERPARAMETER EXPERIMENT COMPARISON


,Experiment,Neurons,Dropout,Learning Rate,Batch Size,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Experiment 5,"(128, 64, 32)",0.25,0.00025,32,0.735523,0.751618,0.692354,0.720770,0.801350
1,Experiment 7,"(128, 64, 32)",0.25,0.00050,64,0.735229,0.753758,0.687584,0.719152,0.802114
2,Experiment 6,"(256, 128, 64)",0.25,0.00050,32,0.734641,0.747523,0.697272,0.721524,0.801238
3,Experiment 2,"(128, 64, 32)",0.25,0.00050,32,0.734274,0.749476,0.692503,0.719864,0.801828
4,Experiment 1,"(128, 64, 32)",0.20,0.00050,32,0.734053,0.750812,0.689372,0.718782,0.801921
5,Experiment 4,"(128, 64, 32)",0.25,0.00100,32,0.733025,0.755227,0.678343,0.714723,0.801566
6,Experiment 3,"(128, 64, 32)",0.30,0.00050,32,0.732804,0.750367,0.686391,0.716955,0.800862


In [14]:
# ==========================================================
# BEST HYPERPARAMETER CONFIGURATION
# ==========================================================

best_result = results_df.iloc[0]

print("=" * 60)
print("BEST HYPERPARAMETER CONFIGURATION")
print("=" * 60)

print("Experiment       :", best_result["Experiment"])
print("Neurons          :", best_result["Neurons"])
print("Dropout          :", best_result["Dropout"])
print("Learning Rate    :", best_result["Learning Rate"])
print("Batch Size       :", best_result["Batch Size"])

print("\nPerformance:")

print(
    "Accuracy         :",
    f"{best_result['Accuracy'] * 100:.2f}%"
)

print(
    "Precision        :",
    f"{best_result['Precision']:.4f}"
)

print(
    "Recall           :",
    f"{best_result['Recall']:.4f}"
)

print(
    "F1 Score         :",
    f"{best_result['F1 Score']:.4f}"
)

print(
    "ROC AUC          :",
    f"{best_result['ROC AUC']:.4f}"
)

BEST HYPERPARAMETER CONFIGURATION
Experiment       : Experiment 5
Neurons          : (128, 64, 32)
Dropout          : 0.25
Learning Rate    : 0.00025
Batch Size       : 32

Performance:
Accuracy         : 73.55%
Precision        : 0.7516
Recall           : 0.6924
F1 Score         : 0.7208
ROC AUC          : 0.8014


In [15]:
# ==========================================================
# FINAL BEST MODEL EVALUATION
# ==========================================================

y_prob_final = best_model.predict(
    X_test,
    verbose=0
).ravel()

y_pred_final = (y_prob_final >= 0.5).astype(int)

final_accuracy = accuracy_score(
    y_test,
    y_pred_final
)

final_precision = precision_score(
    y_test,
    y_pred_final
)

final_recall = recall_score(
    y_test,
    y_pred_final
)

final_f1 = f1_score(
    y_test,
    y_pred_final
)

final_auc = roc_auc_score(
    y_test,
    y_prob_final
)

print("=" * 60)
print("FINAL BEST ANN RESULTS")
print("=" * 60)

print("Accuracy  :", f"{final_accuracy * 100:.2f}%")
print("Precision :", f"{final_precision:.4f}")
print("Recall    :", f"{final_recall:.4f}")
print("F1 Score  :", f"{final_f1:.4f}")
print("ROC AUC   :", f"{final_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

FINAL BEST ANN RESULTS
Accuracy  : 73.55%
Precision : 0.7516
Recall    : 0.6924
F1 Score  : 0.7208
ROC AUC   : 0.8014

Confusion Matrix:
[[5364 1535]
 [2064 4645]]


In [16]:
# ==========================================================
# CLASSIFICATION REPORT
# ==========================================================

print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        y_pred_final,
        target_names=[
            "No Heart Disease",
            "Heart Disease"
        ]
    )
)

CLASSIFICATION REPORT
                  precision    recall  f1-score   support

No Heart Disease       0.72      0.78      0.75      6899
   Heart Disease       0.75      0.69      0.72      6709

        accuracy                           0.74     13608
       macro avg       0.74      0.73      0.73     13608
    weighted avg       0.74      0.74      0.73     13608



In [17]:
# ==========================================================
# SAVE FINAL MODEL AND SCALER
# ==========================================================

os.makedirs("../models", exist_ok=True)

model_path = "../models/heart_disease_ann_final.keras"
scaler_path = "../models/robust_scaler.pkl"

# Save model
best_model.save(model_path)

# Save scaler
joblib.dump(
    scaler,
    scaler_path
)

print("=" * 60)
print("FINAL MODEL SAVED SUCCESSFULLY")
print("=" * 60)

print("Model  :", model_path)
print("Scaler :", scaler_path)

FINAL MODEL SAVED SUCCESSFULLY
Model  : ../models/heart_disease_ann_final.keras
Scaler : ../models/robust_scaler.pkl
